# LUCID — Kaggle Community Benchmarks transport (M01)

This notebook is a **transport surface** for the locked **1.1.0** line.

- One **main** leaderboard task is selected via `%choose` in the **final code cell**.
- Dataset-style evaluation uses a small deterministic slice aligned with `tests/fixtures/kaggle_transport/transport_manifest.json`.

**Install note:** on Kaggle, install this repository before running (example):

`pip install -q "git+https://github.com/m-cahill/lucid.git@main"`


In [ ]:
# If running on Kaggle, install pinned LUCID from GitHub (replace ref as needed).
# Distribution name: lucid-benchmark (`pip show lucid-benchmark`). Import: lucid / lucid.kaggle.
# !pip install -q "git+https://github.com/m-cahill/lucid.git@main"

import importlib.util

_kg = importlib.util.find_spec("lucid.kaggle")
assert _kg is not None, (
    "lucid.kaggle missing: pin a ref that includes src/lucid/kaggle, check pip install, "
    "and remove any partial lucid/ tree shadowing site-packages (see README)."
)

import pandas as pd
import kaggle_benchmarks as kbench

from lucid.families.symbolic_negation_v1 import generate_episode
from lucid.kaggle.episode_llm import score_episode_with_llm
from lucid.models import DriftSeverity


@kbench.task(name="lucid_symbolic_negation_row")
def lucid_symbolic_negation_row(llm, generation_seed: int, drift_severity: str) -> float:
    sev = DriftSeverity[drift_severity]
    spec = generate_episode(seed=int(generation_seed), drift_severity=sev)
    score = score_episode_with_llm(llm, spec)
    return float(score.lucid_score_episode)


_EVAL_DF = pd.DataFrame(
    [
        {"generation_seed": 100, "drift_severity": "LOW"},
        {"generation_seed": 42, "drift_severity": "MEDIUM"},
        {"generation_seed": 200, "drift_severity": "HIGH"},
    ]
)


@kbench.task(name="lucid_main_task")
def lucid_main_task(llm) -> float:
    results = lucid_symbolic_negation_row.evaluate(llm=[llm], evaluation_data=_EVAL_DF)
    df = results.as_dataframe()
    # Robust aggregation: mean of the first numeric column (per-row task return).
    num = df.select_dtypes("number")
    if num.shape[1] < 1:
        raise RuntimeError("No numeric columns returned from evaluation dataframe.")
    return float(num.iloc[:, 0].mean())


In [ ]:
%choose lucid_main_task
